# HomeReach Melbourne — 05: Audit the transport schedule snapshot

**Stage:** inspect the uploaded `gtfs.zip` before cleaning stops or measuring public-transport access.
**File:** 229,799,804 bytes; reviewed SHA-256 is checked below.
**Detailed audit scope:** operational feeds 2 (metropolitan train), 3 (metropolitan tram), and 4 (Myki bus).
The other five feeds are inventoried, not validated or included in these metrics.

The bus feed also includes regional-town services. The train operational feed includes replacement-bus routes.
**None of today's counts is an LGA service score or a count of available vehicles.**
We have not spatially filtered stops to the 31 rental LGAs.

Use the existing **homereach** notebook kernel. Run each cell in order using its ▶ button or **Ctrl+Enter**.
This notebook imports only packages already in the environment (pandas and NumPy, plus Python's standard library).
It does not call an API, download data, change the registry, run Git, or alter the app.

There are three full `stop_times.txt` scans, one per selected feed. They read **100,000-row chunks** and print progress.
Do not press Run repeatedly while a scan is working. Depending on your computer, the bus scan may take a few minutes.
We keep only summaries, sets of referenced IDs and compact duplicate-check state, not the 11.85 million rows in memory.

**Outputs only:** `data/interim/gtfs_audit/` and generated `docs/gtfs_audit.md`.
The raw ZIP and upstream rental/geography files remain unchanged.
Rerunning the export cell replaces the generated audit outputs; keep personal notes in a separate progress document.

Sources:
- Transport Victoria: https://opendata.transport.vic.gov.au/dataset/gtfs-schedule
- GTFS reference: https://gtfs.org/documentation/schedule/reference/
- Extended route codes: https://developers.google.com/transit/gtfs/reference/extended-route-types

## 1. Imports, project path and audit settings

The first message appears before importing packages, making kernel startup easier to distinguish from file processing.
`FEED_ID` is an operational-feed identifier. It is **not** the same thing as a GTFS `route_type`.
The proposed analysis date is fixed for reproducibility, not taken from today's computer clock.

In [ ]:
print("Starting the GTFS audit imports...", flush=True)

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from contextlib import contextmanager
from collections import Counter
from itertools import combinations
from functools import lru_cache
from datetime import datetime, date, timedelta, timezone
import hashlib
import io
import json
import re
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def find_project_root(start):
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / "pyproject.toml").is_file() and (folder / "data").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook inside the existing HomeReach project.")

ROOT = find_project_root(Path.cwd())
INPUT = ROOT / "data/raw/transport/gtfs.zip"
EXPECTED_SHA256 = "c88af3abcaf0872e45196dcc1d5af9bc20a82c54a00a9dfef46c809ec2e34814"
FEED_LABELS = {
    "1": "Regional train",
    "2": "Metropolitan train operational feed",
    "3": "Metropolitan tram",
    "4": "Myki bus: metropolitan + regional town",
    "5": "Regional coach",
    "6": "Regional bus",
    "10": "Interstate",
    "11": "SkyBus",
}
SELECTED = ("2", "3", "4")
ANALYSIS_DATE = date(2026, 10, 13)  # Proposed weekday within this snapshot, not a disruption-free claim.
CHUNK_ROWS = 100_000
PUBLISHER_PAGE = "https://opendata.transport.vic.gov.au/dataset/gtfs-schedule"
GTFS_REFERENCE = "https://gtfs.org/documentation/schedule/reference/"
EXTENDED_TYPES = "https://developers.google.com/transit/gtfs/reference/extended-route-types"

EXPECTED = {
    "2": {"stops": 2859, "routes": 36, "trips": 40556, "calendar": 26, "calendar_dates": 40,
          "stop_time_rows": 688380, "date_trips": 2828, "date_routes": 19},
    "3": {"stops": 1623, "routes": 24, "trips": 55599, "calendar": 12, "calendar_dates": 46,
          "stop_time_rows": 2568892, "date_trips": 5044, "date_routes": 24},
    "4": {"stops": 22633, "routes": 762, "trips": 215328, "calendar": 5823, "calendar_dates": 88898,
          "stop_time_rows": 8589911, "date_trips": 27708, "date_routes": 653},
}
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 65)

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Project root:", ROOT)
print("Detailed feeds:", ", ".join(SELECTED))
print("Proposed analysis service date:", ANALYSIS_DATE.isoformat())

## 2. Verify the source fingerprint and outer archive

The archive contains **eight inner ZIPs**. Each is called `google_transit.zip` inside a different numbered folder.
They must retain their feed identities: extracting them all to the same directory could overwrite equal filenames.

We read one selected inner ZIP at a time. The largest selected compressed inner archive is about 77 MiB.
The outer archive is never extracted or modified. ZIP member timestamps are stored as observed metadata,
not asserted to be a download time or a timezone-aware publication time.

In [ ]:
require(INPUT.is_file(), f"Place the original gtfs.zip at {INPUT}")
print("Checking the 219 MiB source fingerprint...", flush=True)
require(sha256_file(INPUT) == EXPECTED_SHA256,
        "GTFS fingerprint differs from the inspected upload. Re-audit the new file; do not bypass the hash.")
require(INPUT.stat().st_size == 229799804, "Unexpected archive size.")

with ZipFile(INPUT) as outer:
    members = outer.infolist()
    require(len({item.filename for item in members}) == len(members), "Duplicate outer member names.")
    expected_members = {f"{fid}/google_transit.zip" for fid in FEED_LABELS}
    require({item.filename for item in members if not item.is_dir()} == expected_members,
            "The operational-feed inventory differs from the reviewed snapshot.")
    outer_rows = []
    for item in members:
        require(not item.flag_bits & 1, "Encrypted archive member is not supported.")
        fid = item.filename.split("/")[0]
        outer_rows.append({
            "feed_id": fid, "operational_group": FEED_LABELS[fid],
            "member": item.filename, "inner_zip_bytes": item.file_size,
            "outer_compressed_bytes": item.compress_size,
            "zip_member_timestamp_no_timezone": datetime(*item.date_time).isoformat(),
            "detailed_audit": fid in SELECTED,
        })
outer_inventory = pd.DataFrame(outer_rows).sort_values("feed_id", key=lambda s: s.astype(int))

# Record hashes only for existing inputs. No modification of these paths occurs.
tracked_paths = [INPUT]
for relative in [
    "config/rent_snapshot.json", "config/lga_snapshot.json", "docs/data_registry.csv",
    "data/processed/rent_quarterly.csv", "data/processed/geography/metro_lga_2025.gpkg",
]:
    candidate = ROOT / relative
    if candidate.is_file():
        tracked_paths.append(candidate)
upstream_hashes = {path: sha256_file(path) for path in tracked_paths}

print("SOURCE CHECKPOINT PASSED")
print("Outer archive members:", len(outer_inventory))
display(outer_inventory[["feed_id", "operational_group", "inner_zip_bytes", "detailed_audit"]])

## 3. Load the smaller tables, not the entire stop-time tables

All identifiers are read as strings. A numeric-looking ID is a label, not a measurement.
`keep_default_na=False` retains empty fields explicitly. We apply defaults only when interpreting a particular GTFS field.

We fully load agency, stops, routes, trips and both calendar tables for the three selected feeds.
We inventory, but **do not validate the contents of**, shapes, transfers, pathways and levels.
The selected feeds do not contain `feed_info.txt` or `frequencies.txt`.
The absence of `feed_info.txt` here is not by itself a failure.

In [ ]:
@contextmanager
def open_inner(feed_id):
    require(feed_id in FEED_LABELS, f"Unknown operational feed: {feed_id}")
    with ZipFile(INPUT) as outer:
        payload = outer.read(f"{feed_id}/google_transit.zip")
    buffer = io.BytesIO(payload)
    del payload
    try:
        with ZipFile(buffer) as inner:
            require(len(inner.namelist()) == len(set(inner.namelist())), "Duplicate inner archive member.")
            yield inner
    finally:
        buffer.close()

SMALL_FILES = ("agency.txt", "stops.txt", "routes.txt", "trips.txt", "calendar.txt", "calendar_dates.txt")
tables = {}
table_rows = []
for feed_id in SELECTED:
    print(f"Loading metadata: feed {feed_id} — {FEED_LABELS[feed_id]}", flush=True)
    with open_inner(feed_id) as inner:
        require(set(SMALL_FILES).issubset(inner.namelist()), f"Missing a reviewed table in feed {feed_id}.")
        require("stop_times.txt" in inner.namelist(), f"Missing stop_times.txt in feed {feed_id}.")
        require("frequencies.txt" not in inner.namelist(),
                "Frequency-based trips need separate treatment; re-audit the changed feed.")
        for item in inner.infolist():
            table_rows.append({"feed_id": feed_id, "table": item.filename,
                               "uncompressed_bytes": item.file_size, "compressed_bytes": item.compress_size})
        tables[feed_id] = {}
        for name in SMALL_FILES:
            with inner.open(name) as handle:
                frame = pd.read_csv(handle, dtype="string", keep_default_na=False,
                                    encoding="utf-8-sig", low_memory=False)
            tables[feed_id][name] = frame
        # Read only a preview here; the full stop-time scan comes later.
        with inner.open("stop_times.txt") as handle:
            tables[feed_id]["stop_times_preview"] = pd.read_csv(
                handle, dtype="string", keep_default_na=False, encoding="utf-8-sig", nrows=5)

table_inventory = pd.DataFrame(table_rows)
feed_summary = pd.DataFrame([
    {"feed_id": fid, "operational_group": FEED_LABELS[fid],
     **{name[:-4] + "_rows": len(tables[fid][name]) for name in SMALL_FILES}}
    for fid in SELECTED
])
display(feed_summary)
print("Stop-time schema and five-row preview for metropolitan train:")
display(tables["2"]["stop_times_preview"])

## 4. Check metadata keys, references, coordinates and stop hierarchy

A station can contain platforms, entrances and internal pathway nodes. These are separate records in `stops.txt`.
A blank `location_type` means a stop/platform (type 0); it does not mean unknown location type.

The train feed's 2,859 location records are **not 2,859 railway stations**.
We separately record raw wheelchair values: 0/blank is not a blanket claim of inaccessibility.
For child locations, accessibility may inherit from the parent; the audit does not compute accessible journeys.

In [ ]:
DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

def require_key(frame, columns, label):
    require(set(columns).issubset(frame.columns), f"{label}: missing key columns.")
    for column in columns:
        require(frame[column].ne("").all() and frame[column].eq(frame[column].str.strip()).all(),
                f"{label}: blank or space-padded ID.")
    require(not frame.duplicated(columns).any(), f"{label}: duplicate primary key.")

def validate_ymd(series, label):
    require(series.str.fullmatch(r"[0-9]{8}").all(), f"{label}: dates must be YYYYMMDD.")
    for value in series.unique():
        datetime.strptime(value, "%Y%m%d")  # Reject invalid dates without relying on datetime storage units.

metadata_rows, location_rows, route_rows, access_rows, replacement_rows = [], [], [], [], []
replacement_route_ids = {}
for fid in SELECTED:
    t = tables[fid]
    for fname, keys in [
        ("agency.txt", ["agency_id"]), ("stops.txt", ["stop_id"]), ("routes.txt", ["route_id"]),
        ("trips.txt", ["trip_id"]), ("calendar.txt", ["service_id"]),
        ("calendar_dates.txt", ["service_id", "date"]),
    ]:
        require_key(t[fname], keys, f"{fid}/{fname}")
    for stem in ["stops", "routes", "trips", "calendar", "calendar_dates"]:
        require(len(t[stem + ".txt"]) == EXPECTED[fid][stem], f"{fid}/{stem}: changed reviewed row count.")

    stops, routes, trips, cal, exc, agency = [t[name] for name in
        ["stops.txt", "routes.txt", "trips.txt", "calendar.txt", "calendar_dates.txt", "agency.txt"]]
    require(set(agency["agency_timezone"]) == {"Australia/Melbourne"}, "Unexpected agency timezone.")
    require(trips["route_id"].isin(routes["route_id"]).all(), f"{fid}: unknown trip-to-route reference.")
    all_service_ids = set(cal["service_id"]) | set(exc["service_id"])
    require(trips["service_id"].isin(all_service_ids).all(), f"{fid}: undefined trip service ID.")
    require(exc["service_id"].isin(cal["service_id"]).all(), f"{fid}: new exception-only IDs need review.")
    require(routes["agency_id"].isin(set(agency["agency_id"]) | ({""} if len(agency) == 1 else set())).all(),
            f"{fid}: ambiguous or unknown route agency.")
    require((routes["route_short_name"].ne("") | routes["route_long_name"].ne("")).all(),
            f"{fid}: a route has no passenger-facing name.")

    lt = stops["location_type"].replace("", "0")
    require(lt.isin(["0", "1", "2", "3", "4"]).all(), f"{fid}: invalid location_type.")
    require(stops["parent_station"].isin(set(stops["stop_id"]) | {""}).all(), f"{fid}: unknown parent location.")
    parent_types = stops["parent_station"].map(dict(zip(stops["stop_id"], lt)))
    require(parent_types[lt.isin(["2", "3"])].eq("1").all(), f"{fid}: entrance/node parent is not a station.")
    require(stops.loc[lt.eq("1"), "parent_station"].eq("").all(), f"{fid}: station has a parent.")
    optional_child = lt.eq("0") & stops["parent_station"].ne("")
    require(parent_types[optional_child].eq("1").all(), f"{fid}: platform parent is not a station.")
    require(parent_types[lt.eq("4")].eq("0").all(), f"{fid}: boarding-area parent is not a platform.")
    lat = pd.to_numeric(stops["stop_lat"], errors="coerce")
    lon = pd.to_numeric(stops["stop_lon"], errors="coerce")
    require(lat.between(-90, 90).all() and lon.between(-180, 180).all(),
            f"{fid}: missing or invalid coordinates.")
    require(np.isfinite(lat).all() and np.isfinite(lon).all(), f"{fid}: non-finite coordinates.")

    for column in DAYS:
        require(cal[column].isin(["0", "1"]).all(), f"{fid}: invalid weekly service flag.")
    validate_ymd(cal["start_date"], f"{fid}/calendar start")
    validate_ymd(cal["end_date"], f"{fid}/calendar end")
    validate_ymd(exc["date"], f"{fid}/exceptions")
    require(cal["start_date"].le(cal["end_date"]).all(), f"{fid}: reversed calendar interval.")
    require(exc["exception_type"].isin(["1", "2"]).all(), f"{fid}: invalid exception type.")
    for frame, column in [(stops, "wheelchair_boarding"), (trips, "wheelchair_accessible")]:
        require(frame[column].isin(["", "0", "1", "2"]).all(), f"{fid}: unexpected accessibility code.")
        for raw_value, count in frame[column].value_counts().items():
            access_rows.append({"feed_id": fid, "field": column,
                                "raw_value": raw_value if raw_value else "<blank>", "rows": int(count)})

    for raw_value, count in stops["location_type"].value_counts().items():
        location_rows.append({"feed_id": fid, "raw_location_type": raw_value or "<blank>",
                              "interpreted_type": raw_value or "0", "rows": int(count)})
    for route_type, count in routes["route_type"].value_counts().items():
        route_rows.append({"feed_id": fid, "route_type": route_type, "route_records": int(count)})

    # This flag follows an explicit source label, not a general heuristic for all GTFS providers.
    replacement = routes["route_short_name"].str.casefold().eq("replacement bus")
    replacement_route_ids[fid] = set(routes.loc[replacement, "route_id"])
    for row in routes.loc[replacement, ["route_id", "route_short_name", "route_long_name", "route_type"]].to_dict("records"):
        replacement_rows.append({"feed_id": fid, **row})
    metadata_rows.append({
        "feed_id": fid, "primary_keys_unique": True, "required_references_found": True,
        "all_location_coordinates_valid": True, "agency_timezone": "Australia/Melbourne",
        "stop_platform_records": int(lt.eq("0").sum()), "parent_station_records": int(lt.eq("1").sum()),
        "entrance_records": int(lt.eq("2").sum()), "generic_node_records": int(lt.eq("3").sum()),
        "route_records_without_any_trip": len(set(routes["route_id"]) - set(trips["route_id"])),
        "explicit_replacement_route_records": int(replacement.sum()),
    })

metadata_audit = pd.DataFrame(metadata_rows)
location_types = pd.DataFrame(location_rows)
route_types = pd.DataFrame(route_rows)
raw_accessibility = pd.DataFrame(access_rows)
replacement_routes = pd.DataFrame(replacement_rows)
display(metadata_audit)
display(location_types)
display(route_types)
display(raw_accessibility)
print("Explicit Replacement Bus route records in feed 2:", len(replacement_route_ids["2"]))

## 5. Check identifier collisions before ever combining feeds

A `service_id` defines a group of operating dates **inside its own feed**.
Joining across feeds on that raw ID can connect trips to another feed's calendar.

For this snapshot, feeds 2 and 3 share 12 service-ID strings. Tram and Myki bus feeds share 41 stop-ID strings.
A shared string is not automatically a data error, nor proof we should collapse two records.
Later we will use composite keys such as **(feed_id, stop_id)** and **(feed_id, service_id)**.

In [ ]:
collision_rows = []
for left, right in combinations(SELECTED, 2):
    for fname, key in [("stops.txt", "stop_id"), ("routes.txt", "route_id"),
                       ("trips.txt", "trip_id"), ("calendar.txt", "service_id")]:
        overlap = set(tables[left][fname][key]) & set(tables[right][fname][key])
        collision_rows.append({
            "left_feed": left, "right_feed": right, "field": key,
            "shared_raw_id_count": len(overlap), "examples": " | ".join(sorted(overlap)[:5]),
        })
id_collisions = pd.DataFrame(collision_rows)
display(id_collisions.loc[id_collisions["shared_raw_id_count"].gt(0)])
require(len(set(tables["2"]["calendar.txt"]["service_id"]) &
            set(tables["3"]["calendar.txt"]["service_id"])) == 12, "Reviewed service-ID overlap changed.")
require(len(set(tables["3"]["stops.txt"]["stop_id"]) &
            set(tables["4"]["stops.txt"]["stop_id"])) == 41, "Reviewed stop-ID overlap changed.")

## 6. Apply calendar exceptions and inspect the actual dates

Weekly service flags are only the starting point.
For each service day, take weekly active IDs, add exception type 1, then remove exception type 2.

The selected calendar tables span **2026-10-08 to 2026-12-31 (85 days)**.
Transport Victoria describes a rolling 30-day publication window and warns that coverage may be incomplete.
These observations are not the same claim: **calendar endpoints do not guarantee complete service for the whole span**.
We document the observed longer range rather than discard rows to force it to 30 days.

Daily counts below are active `trip_id` records for each **whole operational feed**, after calendar exceptions.
They are not stop-level departures, vehicles, passengers or counts confined to our 31 LGAs.
No `frequencies.txt` exists in the selected feeds, so no headway-expansion calculation is needed here.

In [ ]:
def active_service_ids(calendar, exceptions, service_date):
    if isinstance(service_date, str):
        service_date = date.fromisoformat(service_date)
    key = service_date.strftime("%Y%m%d")
    weekday = DAYS[service_date.weekday()]
    base = set()
    if calendar is not None and not calendar.empty:
        base = set(calendar.loc[
            calendar["start_date"].le(key) & calendar["end_date"].ge(key) &
            calendar[weekday].eq("1"), "service_id"])
    additions, removals = set(), set()
    if exceptions is not None and not exceptions.empty:
        relevant = exceptions.loc[exceptions["date"].eq(key)]
        additions = set(relevant.loc[relevant["exception_type"].eq("1"), "service_id"])
        removals = set(relevant.loc[relevant["exception_type"].eq("2"), "service_id"])
    return (base | additions) - removals

calendar_rows, daily_rows = [], []
for fid in SELECTED:
    cal, exc, trips = [tables[fid][name] for name in ["calendar.txt", "calendar_dates.txt", "trips.txt"]]
    first = min(cal["start_date"].min(), exc["date"].min())
    last = max(cal["end_date"].max(), exc["date"].max())
    start, end = datetime.strptime(first, "%Y%m%d").date(), datetime.strptime(last, "%Y%m%d").date()
    require(start == date(2026, 10, 8) and end == date(2026, 12, 31), "Observed calendar extent changed.")
    calendar_rows.append({
        "feed_id": fid, "first_calendar_day": start.isoformat(), "last_calendar_day": end.isoformat(),
        "calendar_span_days": (end - start).days + 1,
        "exception_additions": int(exc["exception_type"].eq("1").sum()),
        "exception_removals": int(exc["exception_type"].eq("2").sum()),
    })
    for offset in range((end - start).days + 1):
        day = start + timedelta(days=offset)
        active = active_service_ids(cal, exc, day)
        subset = trips.loc[trips["service_id"].isin(active)]
        daily_rows.append({
            "feed_id": fid, "service_date": day.isoformat(), "weekday": day.strftime("%A"),
            "active_service_ids": len(active), "active_trip_records": len(subset),
            "active_route_ids": int(subset["route_id"].nunique()),
            "explicit_replacement_trip_records": int(subset["route_id"].isin(replacement_route_ids[fid]).sum()),
        })

calendar_coverage = pd.DataFrame(calendar_rows)
daily_services = pd.DataFrame(daily_rows)
display(calendar_coverage)
print("A short common inspection window — counts refer to complete operational feeds:")
display(daily_services.loc[daily_services["service_date"].between("2026-10-09", "2026-10-18")])

## 7. Inspect the proposed weekday: Tuesday, 13 October 2026

This date is close to the publisher's 9 October snapshot and has active services in all three feeds.
It is an **illustrative candidate**, not proof of a typical, holiday-free or disruption-free weekday.
The rail operational feed includes **52 explicitly labelled replacement-route trips** on this date.

Observe the `before_exceptions` and `after_exceptions` columns:
ignoring the tram and bus exceptions would substantially over-count scheduled trips.
We will retain this date, timezone and replacement-service distinction in the next spatial-analysis stage.
Rent still describes 2025Q3; transport describes this 2026 service date.

In [ ]:
selected_rows = []
for fid in SELECTED:
    cal, exc, trips = [tables[fid][name] for name in ["calendar.txt", "calendar_dates.txt", "trips.txt"]]
    day_key = ANALYSIS_DATE.strftime("%Y%m%d")
    base_ids = set(cal.loc[
        cal["start_date"].le(day_key) & cal["end_date"].ge(day_key) &
        cal[DAYS[ANALYSIS_DATE.weekday()]].eq("1"), "service_id"])
    ids = active_service_ids(cal, exc, ANALYSIS_DATE)
    active_trips = trips.loc[trips["service_id"].isin(ids)]
    row = {
        "feed_id": fid, "service_date": ANALYSIS_DATE.isoformat(), "timezone": "Australia/Melbourne",
        "trip_records_before_exceptions": int(trips["service_id"].isin(base_ids).sum()),
        "trip_records_after_exceptions": len(active_trips),
        "active_route_ids": int(active_trips["route_id"].nunique()),
        "explicit_replacement_trip_records": int(active_trips["route_id"].isin(replacement_route_ids[fid]).sum()),
    }
    require(row["trip_records_after_exceptions"] == EXPECTED[fid]["date_trips"], "Selected-date trip count changed.")
    require(row["active_route_ids"] == EXPECTED[fid]["date_routes"], "Selected-date route count changed.")
    selected_rows.append(row)
selected_date = pd.DataFrame(selected_rows)
display(selected_date)
require(int(selected_date.loc[selected_date.feed_id.eq("2"), "explicit_replacement_trip_records"].iloc[0]) == 52,
        "Replacement-route inventory changed.")
print("These are raw scheduled trip records, not counts of trains/buses physically operating at once.")

## 8. Define the bounded-memory stop-time scan and test the helpers

A stop-time row is **one trip's visit to one stop**, not one vehicle and not necessarily a useful boarding opportunity.
A final stop has no onward journey on that trip even when its raw pickup flag is 0.
Before computing passenger service frequency later, we must handle terminal records and pickup restrictions.

GTFS allows service-day times beyond 24:00:00. For example, 25:15:30 means 01:15:30 on the following calendar day
within the original service day. We convert times to elapsed seconds, not ordinary clock-time objects.

The scan validates every selected stop-time row for references, time syntax, within-row arrival/departure order,
pickup/drop-off enums, and duplicate **(trip_id, stop_sequence)** keys, including keys split across chunks.
It also checks that every selected trip has a stop-time record and every referenced location is a stop/platform.

**Scope limit:** this is not a complete GTFS conformance validator. It does not validate shape paths, all inter-stop
time progression, transfers, pathways, whole-journey accessibility, vehicle identities, or actual operations.

In [ ]:
TIME_RE = re.compile(r"([0-9]{1,2}):([0-9]{2}):([0-9]{2})")

@lru_cache(maxsize=150_000)
def gtfs_seconds(text):
    # Blank and invalid are kept distinct for the audit, never converted to midnight.
    if text == "":
        return -1
    match = TIME_RE.fullmatch(text)
    if match is None:
        return -2
    hours, minutes, seconds = map(int, match.groups())
    if minutes >= 60 or seconds >= 60:
        return -2
    return hours * 3600 + minutes * 60 + seconds

class SequenceKeys:
    """Exact duplicate tracking across chunks, without storing millions of ID tuples.

    For ordinary sequence values a Python integer stores the seen positions as bits.
    Very large sequence values use a sparse set instead of allocating an enormous bitset.
    Records need not be contiguous or sorted in the input file for this duplicate check.
    """
    def __init__(self):
        self.by_trip = {}
        self.large_keys = set()

    def add_batch(self, trip_ids, sequences):
        duplicate_rows = 0
        for trip_id, sequence in zip(trip_ids, sequences):
            n = int(sequence)
            if n <= 65535:
                bit = 1 << n
                old = self.by_trip.get(trip_id, 0)
                duplicate_rows += int(bool(old & bit))
                self.by_trip[trip_id] = old | bit
            else:
                key = (trip_id, n)
                duplicate_rows += int(key in self.large_keys)
                self.large_keys.add(key)
        return duplicate_rows

def scan_feed_stop_times(feed_id):
    t = tables[feed_id]
    known_trips = set(t["trips.txt"]["trip_id"])
    known_stops = set(t["stops.txt"]["stop_id"])
    seen_trips, seen_stops = set(), set()
    sequence_keys = SequenceKeys()
    counts = Counter({
        key: 0 for key in [
            "stop_time_rows", "unknown_trip_rows", "unknown_stop_rows",
            "blank_arrival_times", "invalid_arrival_times", "blank_departure_times",
            "invalid_departure_times", "departure_at_or_after_24h_rows",
            "arrival_after_departure_rows", "duplicate_trip_sequence_rows",
            "invalid_sequence_rows", "invalid_pickup_rows", "invalid_drop_off_rows",
        ]
    })
    pickups, dropoffs = Counter(), Counter()
    maximum_departure = -1
    started = time.monotonic()
    print(f"Scanning feed {feed_id}: {FEED_LABELS[feed_id]}", flush=True)
    fields = ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time",
              "pickup_type", "drop_off_type"]

    with open_inner(feed_id) as inner:
        with inner.open("stop_times.txt") as handle:
            chunks = pd.read_csv(handle, usecols=fields, dtype="string", keep_default_na=False,
                                 encoding="utf-8-sig", chunksize=CHUNK_ROWS)
            for chunk_number, chunk in enumerate(chunks, 1):
                if chunk.empty:
                    continue
                counts["stop_time_rows"] += len(chunk)
                trip_ids = chunk["trip_id"].tolist()
                stop_ids = chunk["stop_id"].tolist()
                seen_trips.update(trip_ids)
                seen_stops.update(stop_ids)
                counts["unknown_trip_rows"] += int((~chunk["trip_id"].isin(known_trips)).sum())
                counts["unknown_stop_rows"] += int((~chunk["stop_id"].isin(known_stops)).sum())

                arrival = chunk["arrival_time"].map(gtfs_seconds).to_numpy(dtype="int64")
                departure = chunk["departure_time"].map(gtfs_seconds).to_numpy(dtype="int64")
                counts["blank_arrival_times"] += int((arrival == -1).sum())
                counts["invalid_arrival_times"] += int((arrival == -2).sum())
                counts["blank_departure_times"] += int((departure == -1).sum())
                counts["invalid_departure_times"] += int((departure == -2).sum())
                counts["departure_at_or_after_24h_rows"] += int((departure >= 86400).sum())
                counts["arrival_after_departure_rows"] += int(
                    ((arrival >= 0) & (departure >= 0) & (arrival > departure)).sum())
                maximum_departure = max(maximum_departure, int(departure.max()))

                # Integer sequence syntax is checked before safe conversion.
                valid_seq = chunk["stop_sequence"].str.fullmatch(r"[0-9]+")
                sequences = pd.to_numeric(chunk["stop_sequence"], errors="coerce").to_numpy(dtype=float)
                valid_seq = valid_seq.to_numpy(dtype=bool) & np.isfinite(sequences) & (sequences <= 2**31 - 1)
                counts["invalid_sequence_rows"] += int((~valid_seq).sum())
                if valid_seq.any():
                    ids_valid = np.asarray(trip_ids, dtype=object)[valid_seq].tolist()
                    counts["duplicate_trip_sequence_rows"] += sequence_keys.add_batch(
                        ids_valid, sequences[valid_seq].astype("int64"))
                pickups.update(chunk["pickup_type"].tolist())
                dropoffs.update(chunk["drop_off_type"].tolist())
                counts["invalid_pickup_rows"] += int((~chunk["pickup_type"].isin(["", "0", "1", "2", "3"])).sum())
                counts["invalid_drop_off_rows"] += int((~chunk["drop_off_type"].isin(["", "0", "1", "2", "3"])).sum())

                if chunk_number == 1 or chunk_number % 10 == 0:
                    print(f"  {counts['stop_time_rows']:,} rows checked; "
                          f"{time.monotonic() - started:.1f} seconds", flush=True)

    stops = t["stops.txt"]
    selected_stops = stops.loc[stops["stop_id"].isin(seen_stops)]
    seconds = max(maximum_departure, 0)
    result = {
        "feed_id": feed_id, **{name: int(value) for name, value in counts.items()},
        "trip_records_referenced": len(seen_trips),
        "trip_records_without_stop_times": len(known_trips - seen_trips),
        "stop_ids_referenced": len(seen_stops),
        "referenced_non_platform_records": int((~selected_stops["location_type"].isin(["", "0"])).sum()),
        "maximum_departure": f"{seconds // 3600:02}:{seconds % 3600 // 60:02}:{seconds % 60:02}",
        "pickup_type_counts": json.dumps(dict(sorted(pickups.items())), sort_keys=True),
        "drop_off_type_counts": json.dumps(dict(sorted(dropoffs.items())), sort_keys=True),
    }
    require(result["stop_time_rows"] == EXPECTED[feed_id]["stop_time_rows"],
            f"Feed {feed_id}: unexpected complete stop-time row count.")
    print(f"Feed {feed_id} finished: {result['stop_time_rows']:,} rows.", flush=True)
    return result

# Five independent helper-test groups; fictional inputs never enter the real source tables.
dummy_calendar = pd.DataFrame([{
    "service_id": "weekday", **{d: "1" if i < 5 else "0" for i, d in enumerate(DAYS)},
    "start_date": "20261012", "end_date": "20261016",
}], dtype="string")
empty_exceptions = pd.DataFrame(columns=["service_id", "date", "exception_type"], dtype="string")
require(active_service_ids(dummy_calendar, empty_exceptions, "2026-10-12") == {"weekday"} and
        active_service_ids(dummy_calendar, empty_exceptions, "2026-10-16") == {"weekday"} and
        active_service_ids(dummy_calendar, empty_exceptions, "2026-10-17") == set(),
        "Calendar boundary/weekend self-check failed.")  # Group 1

dummy_exceptions = pd.DataFrame([
    {"service_id": "weekday", "date": "20261013", "exception_type": "2"},
    {"service_id": "special", "date": "20261013", "exception_type": "1"},
], dtype="string")
require(active_service_ids(dummy_calendar, dummy_exceptions, "2026-10-13") == {"special"},
        "Exception override self-check failed.")  # Group 2
require(active_service_ids(None, dummy_exceptions, "2026-10-13") == {"special"},
        "Exception-only calendar self-check failed.")  # Group 3
require(gtfs_seconds("25:15:30") == 90930 and gtfs_seconds("32:47:00") == 118020 and
        gtfs_seconds("07:60:00") == -2 and gtfs_seconds("") == -1,
        "GTFS time self-check failed.")  # Group 4
tracker = SequenceKeys()
require(tracker.add_batch(["A", "A", "B"], [1, 4, 1]) == 0 and
        tracker.add_batch(["A", "A", "B"], [4, 2, 65536]) == 1 and
        tracker.add_batch(["B"], [65536]) == 1,
        "Cross-chunk duplicate self-check failed.")  # Group 5
del tracker
print("Five helper self-checks passed: calendar bounds, exceptions, exception-only service, times, duplicate keys.")

## 9. Scan metropolitan-train operational stop times

This scans 688,380 rows, including the operational feed's replacement services.
Do not label all of these rows rail departures; mode classification and terminal filtering are later steps.

In [ ]:
stop_time_results = {}
stop_time_results["2"] = scan_feed_stop_times("2")
display(pd.DataFrame([stop_time_results["2"]]))

## 10. Scan metropolitan-tram stop times

This scans 2,568,892 rows. Progress appears in the cell output.
The feed's all-zero stop wheelchair flags mean no stop-accessibility information here, not zero accessible stops.

In [ ]:
stop_time_results["3"] = scan_feed_stop_times("3")
display(pd.DataFrame([stop_time_results["3"]]))

## 11. Scan the Myki-bus stop times

This is the largest table: **8,589,911 rows**, about 652 MiB uncompressed.
It is read in chunks and is not loaded as one enormous DataFrame.
Some bus locations are outside the study LGAs; the geographical filter comes in the next notebook.

The counter continues to update during the scan. Wait for the cell to finish before running the next cell.

In [ ]:
stop_time_results["4"] = scan_feed_stop_times("4")
display(pd.DataFrame([stop_time_results["4"]]))

## 12. Reconcile the full scans and record interpretation cautions

Zero failures in these checks means the specified fields and relationships passed this audit.
It does not certify that every GTFS rule is satisfied or that actual services run exactly as scheduled.

The source has **250,932 departure-time records at or after 24:00:00** across the three feeds.
They are valid extended service-day times, not 250,932 distinct vehicles or daily night services.

In [ ]:
require(set(stop_time_results) == set(SELECTED), "Run all three stop-time scan cells first.")
stop_time_audit = pd.DataFrame([stop_time_results[fid] for fid in SELECTED])
ZERO_FIELDS = [
    "unknown_trip_rows", "unknown_stop_rows", "blank_arrival_times", "invalid_arrival_times",
    "blank_departure_times", "invalid_departure_times", "arrival_after_departure_rows",
    "invalid_sequence_rows", "duplicate_trip_sequence_rows", "invalid_pickup_rows",
    "invalid_drop_off_rows", "trip_records_without_stop_times", "referenced_non_platform_records",
]
require(stop_time_audit[ZERO_FIELDS].eq(0).all().all(),
        "A stop-time check found an issue. Preserve the output and investigate before proceeding.")
require(int(stop_time_audit["stop_time_rows"].sum()) == 11847183, "Unexpected total checked rows.")
require(int(stop_time_audit["departure_at_or_after_24h_rows"].sum()) == 250932, "Extended-time inventory changed.")
require(stop_time_audit.set_index("feed_id")["trip_records_referenced"].to_dict()
        == {fid: EXPECTED[fid]["trips"] for fid in SELECTED}, "Trip coverage mismatch.")
print("Full selected stop-time rows checked:", int(stop_time_audit["stop_time_rows"].sum()))
display(stop_time_audit[[
    "feed_id", "stop_time_rows", "trip_records_referenced", "stop_ids_referenced",
    "duplicate_trip_sequence_rows", "unknown_trip_rows", "unknown_stop_rows",
    "departure_at_or_after_24h_rows", "maximum_departure",
]])

review_notes = pd.DataFrame([
    {"topic": "Operational scope", "finding": "Detailed audit covers feeds 2, 3 and 4 only.",
     "next_action": "Report omitted regional/interstate/airport feeds; do not call this all transport."},
    {"topic": "Study area", "finding": "Myki bus includes regional-town services; no LGA filtering performed.",
     "next_action": "Spatially join stop/platform points to the validated 31-LGA geometry."},
    {"topic": "Replacement services", "finding": "Feed 2 has 18 labelled replacement routes; 52 related active trips on 2026-10-13.",
     "next_action": "Keep replacement-route trips distinct; feed 2 and route_type 400 do not guarantee a train."},
    {"topic": "Identifier namespaces", "finding": "12 shared rail/tram service IDs; 41 shared tram/bus stop IDs.",
     "next_action": "Use (feed_id, raw_id); do not collapse cross-feed records automatically."},
    {"topic": "Location hierarchy", "finding": "Train locations include platforms, stations, entrances and generic nodes.",
     "next_action": "Do not count all stops.txt rows as stations or boarding stops."},
    {"topic": "Wheelchair information", "finding": "All selected tram and bus wheelchair_boarding values are 0; train values vary.",
     "next_action": "Preserve unknowns and parent-inheritance semantics; no accessible-journey score yet."},
    {"topic": "Service dates", "finding": "Calendar extent is 85 days, longer than the publisher's stated rolling 30-day window.",
     "next_action": "Use the audited nearby candidate date; do not infer complete timetable coverage from endpoints."},
    {"topic": "Boardable services", "finding": "Stop-time rows include terminal visits and some no-pickup records.",
     "next_action": "Filter by active services, pickup policy and onward stop before interpreting boarding frequency."},
    {"topic": "Temporal alignment", "finding": "Rent is 2025Q3; proposed transport service date is 2026-10-13.",
     "next_action": "Display both dates; never insert this current feed into historical rental backtests."},
    {"topic": "Validation limits", "finding": "Shapes, transfers, pathways, all inter-stop time ordering and real operations not validated.",
     "next_action": "Keep the result labelled a scoped schedule audit, not complete GTFS certification."},
])
display(review_notes)

## 13. Export audit evidence and print the checkpoint

This writes small summaries and the generated report, not the full stop-time tables.
Original inputs and existing rental/geography outputs are not changed. The data registry is intentionally
left alone until we have reviewed and implemented the transport preparation pipeline.

The selected-date proposal is preserved in the audit JSON so subsequent work is reproducible.
There is no claim that the longer calendar span is fully reliable or that the chosen date is a normal timetable.

In [ ]:
OUT = ROOT / "data/interim/gtfs_audit"
REPORT = ROOT / "docs/gtfs_audit.md"
require(all(sha256_file(path) == digest for path, digest in upstream_hashes.items()),
        "An upstream input changed during the audit.")
OUT.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

exports = {
    "outer_feed_inventory.csv": outer_inventory,
    "selected_table_inventory.csv": table_inventory,
    "selected_feed_summary.csv": feed_summary,
    "metadata_audit.csv": metadata_audit,
    "location_types.csv": location_types,
    "route_types.csv": route_types,
    "raw_accessibility_counts.csv": raw_accessibility,
    "replacement_routes.csv": replacement_routes,
    "cross_feed_id_collisions.csv": id_collisions,
    "calendar_coverage.csv": calendar_coverage,
    "daily_service_inventory.csv": daily_services,
    "proposed_service_date.csv": selected_date,
    "stop_times_audit.csv": stop_time_audit,
    "review_notes.csv": review_notes,
}
for name, frame in exports.items():
    frame.to_csv(OUT / name, index=False, encoding="utf-8", lineterminator="\n")

summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "filename": INPUT.name, "bytes": INPUT.stat().st_size, "sha256": EXPECTED_SHA256,
        "publisher_page": PUBLISHER_PAGE, "publisher_metadata_date_observed": "2026-10-09",
        "licence_as_listed_by_publisher": "Creative Commons Attribution 4.0",
        "download_date": None,
    },
    "audit_scope": {
        "outer_archives": 8, "detailed_feed_ids": list(SELECTED),
        "deferred_feed_ids": [k for k in FEED_LABELS if k not in SELECTED],
        "not_a_complete_gtfs_validator": True, "lga_filter_applied": False,
    },
    "proposed_service_date": ANALYSIS_DATE.isoformat(),
    "agency_timezone": "Australia/Melbourne",
    "date_selection_note": "Near-snapshot weekday with active services in all three feeds; 52 labelled rail-replacement trips; not certified typical or disruption-free.",
    "calendar_span": {"first": "2026-10-08", "last": "2026-12-31", "days": 85,
                      "warning": "Raw endpoints do not guarantee complete coverage; publisher describes a rolling 30-day window."},
    "selected_date_counts": selected_date.to_dict("records"),
    "stop_time_checks": stop_time_audit.to_dict("records"),
    "helper_self_check_groups": 5,
    "source_unchanged": True,
    "csv_outputs": {name: {"rows": len(frame), "sha256": sha256_file(OUT / name)}
                    for name, frame in exports.items()},
    "software": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
}
(OUT / "audit_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

report_lines = [
    "# HomeReach Melbourne — GTFS snapshot audit", "",
    f"Generated (UTC): {summary['created_at_utc']}", "",
    "## Source and scope", "",
    f"- Source file: `{INPUT.name}`; {INPUT.stat().st_size:,} bytes.",
    f"- SHA-256: `{EXPECTED_SHA256}`.",
    "- Eight numbered inner GTFS archives. Detailed checks cover feeds 2, 3 and 4 only.",
    "- Agency timezone: Australia/Melbourne. No spatial filtering or LGA service score has been produced.",
    "- Original archive and existing rental/geography input files remained unchanged.",
    "- No download date is inferred from filesystem/ZIP timestamps. Publisher page metadata was observed as 9 October 2026.",
    "", "## Selected-feed inventory", "",
    "| Feed | Location rows | Route rows | Trip rows | Stop-time rows checked |",
    "|---|---:|---:|---:|---:|",
]
for fid in SELECTED:
    e = EXPECTED[fid]
    report_lines.append(f"| {fid} — {FEED_LABELS[fid]} | {e['stops']:,} | {e['routes']:,} | {e['trips']:,} | {e['stop_time_rows']:,} |")
report_lines += [
    "", "Counts describe records, not daily vehicles, unique passenger route brands or stations.",
    "", "## Proposed date and calendar effects", "",
    "- Proposed date: **Tuesday, 13 October 2026**, in Australia/Melbourne service-day time.",
    "- Selected calendars span 8 October–31 December 2026 (85 days). This longer raw extent is not a coverage guarantee.",
    "- Transport Victoria describes a rolling 30-day publication window and warns some route information may be incomplete.",
    "- All three feeds have active service IDs on the proposed date; this is not proof it is disruption-free.",
    "", "| Feed | Trips before exceptions | Trips after exceptions | Active route IDs | Labelled replacement trips |",
    "|---|---:|---:|---:|---:|",
]
for r in selected_date.to_dict("records"):
    report_lines.append(
        f"| {r['feed_id']} | {r['trip_records_before_exceptions']:,} | {r['trip_records_after_exceptions']:,} "
        f"| {r['active_route_ids']} | {r['explicit_replacement_trip_records']} |")
report_lines += [
    "", "These are full operational-feed counts. Regional bus records and replacement services have not been removed.",
    "", "## Validation result", "",
    "- 11,847,183 stop-time rows were scanned in chunks; not sampled.",
    "- Duplicate (trip_id, stop_sequence) keys, including across chunks: 0.",
    "- Undefined stop-time trip or stop references: 0.",
    "- Blank/invalid arrival and departure times in selected snapshot: 0.",
    "- Invalid sequences, pickup/drop-off codes and within-row arrival-after-departure values: 0.",
    "- Every selected trip has stop times. Every referenced location is a stop/platform.",
    "- 250,932 departure-time rows use 24:00:00 or later; these are retained as valid service-day times.",
    "- Five helper self-check groups passed; they are notebook checks, not additions to the existing pytest test count.",
    "", "## Decisions and cautions", "",
]
for r in review_notes.to_dict("records"):
    report_lines.append(f"- **{r['topic']}:** {r['finding']} {r['next_action']}")
report_lines += [
    "", "## Next step", "",
    "Review this evidence, then prepare namespaced stop/platform points for the 31-LGA spatial join.",
    "Preserve exact stop IDs, feed IDs, service date, coordinate datum and replacement-route flags.",
    "Do not update scores or the demonstration app from these raw inventory counts.",
    "", "## Reproduce", "",
    "Run `notebooks/05_gtfs_audit.ipynb` with the existing homereach kernel and the unchanged source ZIP.",
    "Detailed tables and audit_summary.json are stored under `data/interim/gtfs_audit/`.",
    "This generated report is replaced on rerun; keep personal reflections elsewhere.",
    "", "## Official references", "",
    f"- {PUBLISHER_PAGE}", f"- {GTFS_REFERENCE}", f"- {EXTENDED_TYPES}", "",
]
REPORT.write_text("\n".join(report_lines), encoding="utf-8")
require(all(sha256_file(path) == digest for path, digest in upstream_hashes.items()),
        "An upstream file changed during export.")

print("GTFS AUDIT CHECKPOINT PASSED")
print("Outer feed archives: 8")
print("Detailed feeds: 2, 3, 4")
print("Stop-time rows checked: 11847183")
print("Duplicate trip/stop-sequence keys: 0")
print("Unknown stop-time trip/stop references: 0")
print("Blank or invalid arrival/departure times: 0")
print("Extended service-day departure rows retained: 250932")
print("Proposed service date: 2026-10-13 (Australia/Melbourne)")
print("Active trip records: feed 2 = 2828; feed 3 = 5044; feed 4 = 27708")
print("Feed 2 labelled replacement trip records on that date: 52")
print("Helper self-check groups: 5 passed")
print("Audit CSV tables saved:", len(exports))
print("Upstream inputs unchanged: True")
print("Saved report:", REPORT.relative_to(ROOT))
print("Saved audit outputs:", OUT.relative_to(ROOT))

## Checkpoint before continuing

Save the notebook with **Ctrl+S**. Read `docs/gtfs_audit.md`.
Send the final **GTFS AUDIT CHECKPOINT PASSED** output or the first error, not screenshots of every cell.

Be able to explain:
- Why `stops.txt` row count is not a station count.
- Why feed ID must remain part of a service/stop identifier.
- Why calendar exceptions must be applied before counting trips.
- Why the train operational feed includes replacement-bus records.
- Why 2025Q3 rents and a 2026 transport date must be displayed separately.

Next: prepare stop/platform records, attach the audited LGA geometry through a spatial join,
and only then calculate scheduled service measures. Do not turn the inventory counts above into a ranking.